# Full V2 BGE-M3 — 371 corpus + 18 development queries
Status **PREPARED — NOT EXECUTED**. Pilih runtime **Python 3.12 + T4 GPU**. Model/revision dan input tidak boleh diganti. Tidak menggunakan venv, apt, CPU fallback, vector lokal, qrels atau query rewriting. Jalankan sembilan cell berurutan. Output nyata perlu review dan import lokal sebelum R3 V2 COMPLETE.

Jika Colab menampilkan too many active sessions, kelola sesi sendiri di menu Manage sessions. Notebook tidak menutup sesi lain. Python 3.13 harus dihentikan; pilih runtime 3.12 yang tersedia, jangan modifikasi Python sistem.


## CELL 1 — Environment Check
Periksa runtime asli. Folder menggunakan exist_ok=True. RESUME=False secara default; aktifkan hanya setelah memeriksa checkpoint INCOMPLETE pada VM dan sesi yang sama. Jika VM dihapus/berganti, mulai run baru; jangan mencampur partial dari sesi lain.


In [ ]:
import sys, os, json, subprocess, importlib.metadata as metadata, uuid
from pathlib import Path
ENVIRONMENT_PASS = False
assert sys.version_info[:2] == (3, 12), 'STOP: pilih Python 3.12 runtime; jangan gunakan apt/venv.'
import torch
assert torch.cuda.is_available(), 'STOP: pilih T4 GPU runtime. CPU fallback tidak diizinkan.'
WORK = Path('/content/wo_bge_m3_full')
WORK.mkdir(parents=True, exist_ok=True)
for name, folder in {'HF_HOME':'hf_cache','HF_HUB_CACHE':'hf_cache/hub','TORCH_HOME':'torch_cache','PIP_CACHE_DIR':'pip_cache','TMP':'tmp','TEMP':'tmp'}.items():
    path=WORK/folder; path.mkdir(parents=True,exist_ok=True); os.environ[name]=str(path)
os.environ['HF_HUB_OFFLINE']='0'; os.environ['TRANSFORMERS_OFFLINE']='0'
SESSION_ID = globals().get('SESSION_ID', uuid.uuid4().hex)
RESUME = False
print('Python:', sys.version, 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), 'free/total VRAM:', torch.cuda.mem_get_info())
for name in ('FlagEmbedding','transformers','tokenizers','sentence-transformers','peft','numpy','huggingface-hub'):
    try: print(name, metadata.version(name))
    except metadata.PackageNotFoundError: print(name, 'NOT INSTALLED')
assert not (WORK/'bge_m3_v2_full_embeddings.zip').exists(), 'STOP: COMPLETE output exists; download/archive and move it explicitly first.'

## CELL 2 — Dependency Setup
Gunakan Torch CUDA bawaan yang telah terdeteksi; jangan menggantinya dengan versi laptop. Core API dipin, Torch/NumPy existing dilindungi constraint, dependency transitif diselesaikan pip. Dry-run dan pip check harus PASS. Jika paket dipasang, cell berhenti dengan instruksi **Runtime > Restart session**, kemudian ulangi Cell 1–2. File kerja tetap tersimpan selama VM dipertahankan. Jangan lanjut Cell 3 sampai ENVIRONMENT PASS. Jangan mengabaikan konflik paket bawaan.


In [ ]:
import importlib.metadata as metadata
PINS={'FlagEmbedding':'1.4.2','transformers':'4.57.6','tokenizers':'0.22.2','sentence-transformers':'5.2.0','peft':'0.18.1'}
def installed(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
try:
    assert installed('torch') == torch.__version__, 'Restart session: loaded Torch differs from installed version.'
    protected={n:installed(n) for n in ('torch','numpy') if installed(n)}
    constraints=WORK/'constraints.txt'
    constraints.write_text(chr(10).join(n+'=='+v for n,v in protected.items())+chr(10))
    needs=[n+'=='+v for n,v in PINS.items() if installed(n)!=v]
    if needs:
        plan=WORK/'dependency_plan.json'
        base=[sys.executable,'-m','pip','install','-c',str(constraints)]
        subprocess.run(base+['--dry-run','--report',str(plan)]+needs,check=True)
        proposals=json.loads(plan.read_text())['install']
        assert all(item['metadata']['name'].lower()!='torch' for item in proposals), 'STOP: resolver attempts Torch replacement.'
        print('Dependency plan:', [(p['metadata']['name'],p['metadata']['version']) for p in proposals])
        subprocess.run(base+needs,check=True)
        subprocess.run([sys.executable,'-m','pip','check'],check=True)
        (WORK/'setup_checkpoint.json').write_text(json.dumps({'status':'RESTART_REQUIRED','protected':protected,'pins':PINS}))
        raise RuntimeError('RESTART REQUIRED: Runtime > Restart session. Setelah restart jalankan Cell 1 dan 2 lagi. Jangan lanjut Cell 3 sekarang.')
    subprocess.run([sys.executable,'-m','pip','check'],check=True)
    from FlagEmbedding import BGEM3FlagModel
    from transformers import AutoTokenizer
    import numpy as np
    assert torch.cuda.is_available()
    for dtype in (torch.float32,torch.float16):
        a=torch.tensor([[1.,2.],[3.,4.]],device='cuda:0',dtype=dtype)
        assert torch.equal(a@a,torch.tensor([[7.,10.],[15.,22.]],device='cuda:0',dtype=dtype)), 'CUDA numerical smoke FAIL'
    del a
    torch.cuda.synchronize()
    ENVIRONMENT_PASS=True
    print('ENVIRONMENT PASS — imports, pip check, CUDA FP32/FP16 smoke')
except BaseException:
    ENVIRONMENT_PASS=False
    print('ENVIRONMENT FAIL / STOP — lihat penyebab di atas; tidak ada fallback otomatis.')
    raise

## CELL 3 — Upload Input
Upload hanya **full_input_package.zip**, tidak perlu corpus asli/model/qrels/indeks. Notebook memeriksa hash ZIP sebelum membaca runtime.


In [ ]:
assert ENVIRONMENT_PASS, 'STOP: Cell 2 belum PASS.'
from google.colab import files
import hashlib,zipfile,io,importlib.util
uploaded=files.upload()
assert set(uploaded)=={'full_input_package.zip'}, 'Upload tepat satu full_input_package.zip'
raw=uploaded['full_input_package.zip']
EXPECTED_ZIP_SHA='634f861031fb002884f5637aebdd0ff2256dbfc1f831fcd155f3c9efcdf6f112'
assert hashlib.sha256(raw).hexdigest()==EXPECTED_ZIP_SHA, 'STOP: input ZIP checksum mismatch.'
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    infos=archive.infolist()
    assert len(infos)==4 and {i.filename for i in infos}=={'inputs.json','runtime.py','exchange.py','shared_text.py'}
    assert all(i.file_size <= 4*1024**2 and not i.flag_bits & 1 for i in infos)
    content={i.filename:archive.read(i) for i in infos}
manifest=json.loads(content['inputs.json'])
assert manifest['package_fingerprint']=='88a40a5ef5ce264ae282c67a90bb79525d34cd5c1eedb11f03fda01341fd554b'
for name in ('runtime.py','exchange.py','shared_text.py'):
    assert hashlib.sha256(content[name]).hexdigest()==manifest['runtime_files_sha256'][name]
    (WORK/name).write_bytes(content[name])
INPUT=WORK/'full_input_package.zip'; INPUT.write_bytes(raw)
if str(WORK) not in sys.path: sys.path.insert(0,str(WORK))
spec=importlib.util.spec_from_file_location('full_runtime',WORK/'runtime.py')
runtime=importlib.util.module_from_spec(spec); spec.loader.exec_module(runtime)
PACKAGE,_=runtime.read_input(INPUT)
assert PACKAGE['counts']['corpus']==371 and PACKAGE['counts']['queries']==18
print('INPUT PASS:',PACKAGE['counts'],PACKAGE['snapshot_id'],PACKAGE['snapshot_fingerprint'])

## CELL 4 — Tokenization Audit
Unduh checkpoint pinned ke cache Colab sekali; rerun menggunakan cache terverifikasi. Belum ada weights loading. Seluruh 389 input harus cocok dengan token count actual dan <=640, termasuk special tokens.


In [ ]:
assert ENVIRONMENT_PASS
if 'RUN' not in globals(): RUN=runtime.CloudRun(PACKAGE,WORK,SESSION_ID)
assert RUN.package==PACKAGE and RUN.session_id==SESSION_ID, 'STOP: stale session/input; jangan reuse RUN berbeda.'
RUN.tokenize()

## CELL 5 — Model Loading
Satu model CUDA FP16 untuk corpus dan query. Partial hanya boleh dilanjutkan dengan RESUME=True yang dipilih eksplisit. Tidak ada retry setelah OOM.


In [ ]:
assert ENVIRONMENT_PASS
assert RESUME or not any((WORK/(kind+'_partial')).exists() for kind in ('corpus','queries')), 'STOP: periksa checkpoint lalu set RESUME=True pada Cell 1.'
if RUN.model is None: RUN.load()
else: print('Model existing dipertahankan; tidak load ulang.')

## CELL 6 — Corpus Embeddings
Sequential batch 1, progress 1/371–371/371. Checkpoint diberi INCOMPLETE. Resume memeriksa sesi, environment, identitas model/input, ID/order, shape dan checksum tiap row. File checkpoint berada di WORK; hilang jika VM dihapus.


In [ ]:
RUN.encode_group('corpus',resume=RESUME)

## CELL 7 — Query Embeddings
18 original draft texts, normalisasi shared existing tanpa prompt/evidence. Model dan sesi sama. Tidak ada relevance labels.


In [ ]:
RUN.encode_group('queries',resume=RESUME)

## CELL 8 — Vector Validation
371×1024 corpus dan18×1024 query normalized float32, seluruh nilai finite, nonzero, unique canonical IDs, tidak ada truncation. Output sebelum project L2 juga disimpan untuk kompatibilitas shared query cache existing.


In [ ]:
RUN.validate()

## CELL 9 — Export
ZIP COMPLETE hanya setelah semua validasi. Tidak overwrite file COMPLETE. Download **bge_m3_v2_full_embeddings.zip**, simpan di D:/ModelCache/wo-retrieval-benchmark/colab_exchange/v2/, lalu kirim untuk review. Import lokal dilakukan setelah persetujuan terpisah. Waktu encoding bukan hasil benchmark retrieval.


In [ ]:
OUTPUT=WORK/'bge_m3_v2_full_embeddings.zip'
RUN.export(OUTPUT)
files.download(str(OUTPUT))